THIS IS PRACTICE FOR THE K-NN ALGORITHM WITH IRIS DATASET 

In [1]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter #doubtful about this
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo 

Fetching the dataset and preprocessing it

In [3]:

def fetch_iris_data():
    # fetch dataset 
    iris = fetch_ucirepo(id=53) 
      
    # data (as pandas dataframes) 
    X = iris.data.features 
    y = iris.data.targets 
  
    # split data into training and testing datasets(80:20)
    x_train , x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # standardize datasets
    scaler =  StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
  
    return x_train , x_test, y_train, y_test, X.columns.tolist(), y['class'].unique()


**Implementation of K-nearest neighbor**

references-
1. https://www.geeksforgeeks.org/python/
2. 

In [4]:
class KNN:
    def __init__(self, k, dist_metric):
        self.k = k
        self.dist_metric = dist_metric
       # self.weighted = weighted
       
    def fit(self, x_train, y_train):
        self.x_train = np.array(x_train)
        self.y_train = np.array(y_train).ravel()

    def cal_distance(self, x1, x2):
        if self.dist_metric == 'euclidean':
            return np.linalg.norm(x1 - x2)
        elif self.dist_metric == 'manhattan':
            return np.sum(np.abs(x1-x2))

    def predict_single_data_point(self, x):
        # Calculate distances to all training data sets
        distance = []
        for i, x_train in enumerate(self.x_train):
            dist =  self.cal_distance(x, x_train)
            distance.append((dist, self.y_train[i]))
        # sort
        distance.sort(key=lambda x: x[0])
        knn = distance[:self.k]

        # get labels of knn
        k_labels = [label for _, label in knn]
        most_common = Counter(k_labels).most_common(1)
        return most_common[0][0]

    def predict(self, x_test):
        x_test = np.array(x_test)
        return np.array([self.predict_single_data_point(x) for x in x_test])

    def checkAccuracy(self, x_test, y_test):
        predictions = self.predict(x_test)
        y_test_array = np.array(y_test).ravel()
        accuracy = np.mean(predictions == y_test_array)
        return accuracy
        
        

evaluate k-values

In [5]:
def evaluate_k_values(x_train, x_test, y_train, y_test, dist_metric, k_values):
    result = {}
    for k in k_values : 
        k_nn = KNN(k=k,dist_metric= dist_metric)
        k_nn.fit(x_train, y_train) 
        accuracy = k_nn.checkAccuracy(x_test, y_test)
        result[k] = accuracy
        print(f"Accuracy for k{k} = {accuracy}")
    return result

In [6]:
def main():
    x_train , x_test, y_train, y_test, feature, target = fetch_iris_data()
     # Define k values to test
    k_values = [1, 3, 5, 7, 11, 15]
    dist_metric = 'euclidean'
    res = evaluate_k_values(x_train, x_test, y_train, y_test, dist_metric, k_values)

if __name__ == "__main__":
    main()

Accuracy for k1 = 0.9666666666666667
Accuracy for k3 = 0.9333333333333333
Accuracy for k5 = 0.9333333333333333
Accuracy for k7 = 0.9666666666666667
Accuracy for k11 = 0.9666666666666667
Accuracy for k15 = 0.9666666666666667
